# S5 Eval Curves: Offline BC MC vs NAIL-OPD MC

This notebook recreates WandB-style eval plots for three noise regimes:

- low noise
- medium noise
- high noise

Each regime gets two plots:

- `iter` vs `val/clean_full_exact`
- `iter` vs `val/clean_final_exact`

Important workflow note:

- the first few code cells are intentionally lightweight
- W&B syncing happens in one dedicated cell later on
- that sync cell is opt-in: leave `ENABLE_WANDB_SYNC = False` for a fast local run, or flip it to `True` only when you want to refresh cache files


In [ ]:
from pathlib import Path

import json
import pandas as pd

# Resolve the repo root whether the notebook is opened from the repo root or the notebooks/ folder.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

WANDB_ENTITY = "vedsriraman-columbia-university"
WANDB_PROJECT = "small-cot-experiments"
WANDB_CACHE_DIR = ROOT / "analysis" / "cache" / "wandb"
WANDB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Keep notebook startup fast by default. Only flip this to True when you explicitly want to refresh W&B cache files.
ENABLE_WANDB_SYNC = False

# Coverage and gap diagnostics are shown for one regime at a time so the notebook stays responsive.
COVERAGE_REGIME = "low"

# Save exported plot images into the repo so they can be embedded in the experiment log markdown.
SAVE_PLOTS = True
PLOT_EXPORT_DIR = ROOT / "analysis" / "figures" / "s5_eval_curves"
PLOT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Offline BC MC runs are sourced from W&B for every eta we want to compare.
OFFLINE_BC_RUNS = {
    0.05: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p05"},
    0.10: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p1"},
    0.20: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p2"},
    0.30: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p3"},
    0.40: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p4"},
    0.50: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p5"},
    0.60: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p6"},
    0.70: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p7"},
    0.80: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p8"},
    0.90: {"run_name": "s5-noisy-bc-sample-then-corrupt-n8000000-eta0p9"},
}

# Low-eta NAIL runs need W&B for the early part of training because the local eval file only contains the resumed tail.
# For eta >= 0.4 the local eval history on blocklab should already contain the full curve.
NAIL_RUNS = {
    0.05: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p05-distributional_noise-t1p0", "needs_wandb": True},
    0.10: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p1-distributional_noise-t1p0", "needs_wandb": True},
    0.20: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p2-distributional_noise-t1p0", "needs_wandb": True},
    0.30: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p3-distributional_noise-t1p0", "needs_wandb": True},
    0.40: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p4-distributional_noise-t1p0", "needs_wandb": False},
    0.50: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p5-distributional_noise-t1p0", "needs_wandb": False},
    0.60: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p6-distributional_noise-t1p0", "needs_wandb": False},
    0.70: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p7-distributional_noise-t1p0", "needs_wandb": False},
    0.80: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p8-distributional_noise-t1p0", "needs_wandb": False},
    0.90: {"run_name": "s5-opd-forward_kl_simple-n8000000-eta0p9-distributional_noise-t1p0", "needs_wandb": False},
}

REGIMES = {
    "low": {
        "title_prefix": "low-noise regime",
        "offline_etas": [0.05, 0.10, 0.20, 0.30],
        "nail_etas": [0.05, 0.10, 0.20, 0.30, 0.40],
    },
    "medium": {
        "title_prefix": "medium-noise regime",
        "offline_etas": [0.40, 0.50, 0.60],
        "nail_etas": [0.40, 0.50, 0.60, 0.70],
    },
    "high": {
        "title_prefix": "high-noise regime",
        "offline_etas": [0.70, 0.80, 0.90],
        "nail_etas": [0.70, 0.80, 0.90],
    },
}

METRICS = ["val/clean_full_exact", "val/clean_final_exact"]
EVAL_KEYS = ["iter", *METRICS]

ROOT


In [ ]:
# Helper utilities stay local and instant; no network calls happen in this cell.

def to_jsonable(obj):
    try:
        json.dumps(obj)
        return obj
    except TypeError:
        if isinstance(obj, dict):
            return {str(k): to_jsonable(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [to_jsonable(v) for v in obj]
        return str(obj)


def eta_tag(eta: float) -> str:
    return str(eta).replace('.', 'p')


def meta_path_for_run_id(run_id: str) -> Path:
    return WANDB_CACHE_DIR / f"{run_id}.meta.json"


def eval_path_for_run_id(run_id: str) -> Path:
    return WANDB_CACHE_DIR / f"{run_id}.eval.csv"


def build_cached_run_name_lookup() -> dict[str, dict]:
    # Read any already-exported W&B metadata from disk so we can avoid unnecessary API searches.
    lookup = {}
    for path in sorted(WANDB_CACHE_DIR.glob("*.meta.json")):
        meta = json.loads(path.read_text(encoding="utf-8"))
        run_name = meta.get("run_name")
        if run_name:
            lookup[run_name] = meta
    return lookup


def normalize_eval_df(df: pd.DataFrame) -> pd.DataFrame:
    # Standardize history tables so all downstream code can assume the same schema.
    expected = ["iter", *METRICS]
    missing = [col for col in expected if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    work = df.loc[:, expected].copy()
    work["iter"] = pd.to_numeric(work["iter"], errors="coerce")
    for metric in METRICS:
        work[metric] = pd.to_numeric(work[metric], errors="coerce")
    return work.dropna(subset=["iter"]).reset_index(drop=True)


def dedupe_by_iter(df: pd.DataFrame) -> pd.DataFrame:
    # When we stitch W&B + local resumed tails together, repeated iters can appear.
    # Sorting by source_priority means the local resumed row wins when both sources contain the same iter.
    if df.empty:
        return df.copy()
    work = df.copy().reset_index(drop=True)
    work["_row_order"] = range(len(work))
    work = work.sort_values(["iter", "source_priority", "_row_order"])
    work = work.groupby("iter", as_index=False).last()
    return work.drop(columns=["_row_order"], errors="ignore").sort_values("iter").reset_index(drop=True)


In [ ]:
# Cache status: this is the quick check before any W&B sync work.
cached_lookup = build_cached_run_name_lookup()

status_rows = []
for eta, info in OFFLINE_BC_RUNS.items():
    meta = cached_lookup.get(info["run_name"])
    status_rows.append(
        {
            "group": "Offline BC MC",
            "eta": eta,
            "run_name": info["run_name"],
            "has_cached_meta": meta is not None,
            "has_cached_eval": False if meta is None else eval_path_for_run_id(meta["run_id"]).exists(),
        }
    )

for eta, info in NAIL_RUNS.items():
    meta = cached_lookup.get(info["run_name"])
    status_rows.append(
        {
            "group": "NAIL-OPD MC",
            "eta": eta,
            "run_name": info["run_name"],
            "needs_wandb": info["needs_wandb"],
            "has_cached_meta": meta is not None,
            "has_cached_eval": False if meta is None else eval_path_for_run_id(meta["run_id"]).exists(),
        }
    )

pd.DataFrame(status_rows).sort_values(["group", "eta"]).reset_index(drop=True)


In [ ]:
# Optional sync cell: run this once if any required W&B histories are missing from cache.
# This is the only cell that may take a while, because it may search W&B and download history CSVs.

if not ENABLE_WANDB_SYNC:
    print("Skipping W&B sync. Set ENABLE_WANDB_SYNC = True in the config cell when you want to refresh cache files.")
else:
    import wandb

    api = wandb.Api(timeout=120)

    needed_run_names = sorted(
        {info["run_name"] for info in OFFLINE_BC_RUNS.values()}
        | {info["run_name"] for info in NAIL_RUNS.values() if info["needs_wandb"]}
    )

    cached_lookup = build_cached_run_name_lookup()
    missing_names = [name for name in needed_run_names if name not in cached_lookup or not eval_path_for_run_id(cached_lookup[name]["run_id"]).exists()]

    print(f"Need to sync {len(missing_names)} W&B runs")
    for name in missing_names:
        print("  ", name)

    if missing_names:
        unresolved = set(missing_names)
        found = {}
        for run in api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}"):
            if run.name not in unresolved:
                continue

            eval_df = pd.DataFrame(run.scan_history(keys=EVAL_KEYS, page_size=1000))
            eval_df.to_csv(eval_path_for_run_id(run.id), index=False)

            meta = {
                "run_id": run.id,
                "run_name": run.name,
                "run_path": "/".join(run.path),
                "url": run.url,
                "state": run.state,
                "summary": to_jsonable(dict(run.summary)),
            }
            meta_path_for_run_id(run.id).write_text(json.dumps(meta, indent=2), encoding="utf-8")

            found[run.name] = run.id
            unresolved.remove(run.name)
            print(f"synced {run.name} -> {run.id} ({len(eval_df)} rows)")

            if not unresolved:
                break

        if unresolved:
            raise RuntimeError(f"Could not find these W&B runs: {sorted(unresolved)}")
    else:
        print("All required W&B caches already exist.")


In [ ]:
# After syncing, rebuild the lookup and define loaders that rely on local cache files.
cached_lookup = build_cached_run_name_lookup()


def cached_run_id_for_name(run_name: str) -> str:
    if run_name not in cached_lookup:
        raise RuntimeError(f"Missing cached W&B metadata for {run_name}. Run the sync cell above first.")
    return cached_lookup[run_name]["run_id"]


def load_offline_bc_curve(eta: float) -> pd.DataFrame:
    # Offline BC MC is sourced entirely from W&B cache.
    run_id = cached_run_id_for_name(OFFLINE_BC_RUNS[eta]["run_name"])
    path = eval_path_for_run_id(run_id)
    df = normalize_eval_df(pd.read_csv(path))
    df["group"] = "Offline BC MC"
    df["eta"] = eta
    df["source"] = f"wandb_csv:{run_id}"
    df["source_priority"] = 0
    df["label"] = f"Offline BC MC, eta={eta:.2f}"
    return dedupe_by_iter(df)


def load_local_nail_curve(eta: float) -> pd.DataFrame:
    # Local NAIL eval files are the authoritative source for the resumed tail and for later regimes.
    path = ROOT / f"out-s5-opd-forward_kl_simple-n8000000-eta{eta_tag(eta)}-distributional_noise-t1p0" / "eval_history.jsonl"
    if not path.exists():
        return pd.DataFrame(columns=["iter", *METRICS, "group", "eta", "source", "source_priority", "label"])
    df = normalize_eval_df(pd.read_json(path, lines=True))
    df["group"] = "NAIL-OPD MC"
    df["eta"] = eta
    df["source"] = str(path.relative_to(ROOT))
    df["source_priority"] = 1
    df["label"] = f"NAIL-OPD MC, eta={eta:.2f}"
    return df


def load_wandb_nail_curve(eta: float) -> pd.DataFrame:
    # Only the low-eta NAIL runs need W&B history for the early optimizer steps.
    if not NAIL_RUNS[eta]["needs_wandb"]:
        return pd.DataFrame(columns=["iter", *METRICS, "group", "eta", "source", "source_priority", "label"])
    run_id = cached_run_id_for_name(NAIL_RUNS[eta]["run_name"])
    path = eval_path_for_run_id(run_id)
    df = normalize_eval_df(pd.read_csv(path))
    df["group"] = "NAIL-OPD MC"
    df["eta"] = eta
    df["source"] = f"wandb_csv:{run_id}"
    df["source_priority"] = 0
    df["label"] = f"NAIL-OPD MC, eta={eta:.2f}"
    return df


def load_nail_curve(eta: float) -> pd.DataFrame:
    # Stitch early W&B history to the local resumed tail, then dedupe repeated iters.
    combined = pd.concat([load_wandb_nail_curve(eta), load_local_nail_curve(eta)], ignore_index=True)
    if combined.empty:
        raise FileNotFoundError(f"No NAIL eval data found for eta={eta}")
    return dedupe_by_iter(combined)


def missing_cached_run_names_for_regime(regime_key: str) -> list[str]:
    # Only the Offline BC curves and low-eta NAIL curves depend on cached W&B exports.
    regime = REGIMES[regime_key]
    needed_run_names = [OFFLINE_BC_RUNS[eta]["run_name"] for eta in regime["offline_etas"]]
    needed_run_names += [
        NAIL_RUNS[eta]["run_name"]
        for eta in regime["nail_etas"]
        if NAIL_RUNS[eta]["needs_wandb"]
    ]

    missing = []
    for run_name in needed_run_names:
        meta = cached_lookup.get(run_name)
        if meta is None or not eval_path_for_run_id(meta["run_id"]).exists():
            missing.append(run_name)
    return missing


def ensure_regime_cache_ready(regime_key: str) -> None:
    missing = missing_cached_run_names_for_regime(regime_key)
    if missing:
        raise RuntimeError(
            f"Missing cached W&B histories for regime '{regime_key}': {missing}. "
            "Set ENABLE_WANDB_SYNC = True in the config cell and rerun the sync cell."
        )


def summarize_coverage(curve_df: pd.DataFrame) -> pd.DataFrame:
    return (
        curve_df.groupby(["group", "eta", "label"], as_index=False)
        .agg(
            n_points=("iter", "size"),
            min_iter=("iter", "min"),
            max_iter=("iter", "max"),
            final_clean_full_exact=("val/clean_full_exact", "last"),
            final_clean_final_exact=("val/clean_final_exact", "last"),
        )
        .sort_values(["group", "eta"])
        .reset_index(drop=True)
    )


def summarize_metric_gap(curve_df: pd.DataFrame) -> pd.DataFrame:
    return (
        curve_df.assign(metric_gap=curve_df["val/clean_final_exact"] - curve_df["val/clean_full_exact"])
        .groupby(["group", "eta", "label"], as_index=False)
        .agg(
            max_abs_gap=("metric_gap", lambda s: float(s.abs().max())),
            final_gap=("metric_gap", "last"),
        )
        .sort_values(["group", "eta"])
        .reset_index(drop=True)
    )


curve_df_cache = {}


def load_curve_df_for_regime(regime_key: str) -> pd.DataFrame:
    # Load only the curves needed for one regime so low-noise work does not wait on medium/high cache state.
    if regime_key in curve_df_cache:
        return curve_df_cache[regime_key]

    ensure_regime_cache_ready(regime_key)
    regime = REGIMES[regime_key]
    curves = []

    print(f"Loading Offline BC MC curves for {regime_key} regime...")
    for eta in regime["offline_etas"]:
        print(f"  Offline BC MC eta={eta:.2f}")
        curves.append(load_offline_bc_curve(eta))

    print(f"Loading NAIL-OPD MC curves for {regime_key} regime...")
    for eta in regime["nail_etas"]:
        print(f"  NAIL-OPD MC eta={eta:.2f}")
        curves.append(load_nail_curve(eta))

    curve_df = pd.concat(curves, ignore_index=True)
    print(
        f"Loaded {len(curve_df)} eval points across "
        f"{curve_df[['group', 'eta']].drop_duplicates().shape[0]} curves for {regime_key} regime."
    )
    curve_df_cache[regime_key] = curve_df
    return curve_df


In [ ]:
# Coverage summary is shown for one regime at a time.
# That keeps this diagnostic cell fast and avoids failing on medium/high caches when you only care about low noise.
curve_df_for_coverage = load_curve_df_for_regime(COVERAGE_REGIME)
coverage_df = summarize_coverage(curve_df_for_coverage)
coverage_df


In [ ]:
# Diagnostic: quantify how different clean_final_exact is from clean_full_exact.
# In low noise these gaps can be tiny, which makes the two plots look almost identical.
metric_gap_df = summarize_metric_gap(curve_df_for_coverage)
metric_gap_df


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FAF7F2",
        "axes.edgecolor": "#333333",
        "axes.grid": True,
        "grid.color": "#D8D2C8",
        "grid.alpha": 0.8,
        "grid.linestyle": "--",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
        "font.size": 12,
        "axes.titlesize": 15,
        "axes.labelsize": 12,
    }
)

# Use an intentionally odd, non-monotone palette so nearby curves are easier to distinguish.
ODD_PALETTE = [
    "#FF006E", "#3A86FF", "#8338EC", "#FB5607", "#2A9D8F",
    "#E76F51", "#6D597A", "#90BE6D", "#F72585", "#4D908E",
    "#F9C74F", "#277DA1", "#B5179E", "#43AA8B", "#F9844A",
    "#577590", "#7209B7", "#00B4D8", "#F94144", "#A7C957",
]

all_labels = [
    *(f"Offline BC MC, eta={eta:.2f}" for eta in sorted(OFFLINE_BC_RUNS)),
    *(f"NAIL-OPD MC, eta={eta:.2f}" for eta in sorted(NAIL_RUNS)),
]
CURVE_COLORS = {label: ODD_PALETTE[i] for i, label in enumerate(all_labels)}


def labels_for_regime(regime_key: str) -> list[str]:
    regime = REGIMES[regime_key]
    return [
        *(f"Offline BC MC, eta={eta:.2f}" for eta in regime["offline_etas"]),
        *(f"NAIL-OPD MC, eta={eta:.2f}" for eta in regime["nail_etas"]),
    ]


def plot_filename(metric: str, regime_key: str) -> str:
    metric_name = metric.split('/')[-1]
    return f"{regime_key}_{metric_name}.png"


def plot_regime(metric: str, regime_key: str) -> None:
    # Shared plotting helper so each regime/metric plot cell stays short and readable.
    curve_df = load_curve_df_for_regime(regime_key)
    regime = REGIMES[regime_key]
    labels = labels_for_regime(regime_key)
    metric_name = metric.split('/')[-1]
    title = f"Offline BC MC vs NAIL-OPD MC {metric_name} in {regime['title_prefix']}"

    fig, ax = plt.subplots(figsize=(16, 5.75), constrained_layout=True)
    for label in labels:
        df = curve_df[curve_df["label"] == label].sort_values("iter")
        ax.plot(
            df["iter"],
            df[metric],
            color=CURVE_COLORS[label],
            marker="o",
            linewidth=2.2,
            markersize=4,
            label=label,
        )

    ax.set_title(title)
    ax.set_xlabel("iter")
    ax.set_ylabel(metric)
    ax.set_ylim(0.0, 1.01)
    ax.xaxis.set_major_locator(MultipleLocator(10000))
    ax.xaxis.set_minor_locator(MultipleLocator(5000))
    ax.grid(which="minor", linestyle=":", alpha=0.35)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

    if SAVE_PLOTS:
        out_path = PLOT_EXPORT_DIR / plot_filename(metric, regime_key)
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
        print(f"Saved plot to {out_path}")

    plt.show()


In [ ]:
plot_regime("val/clean_full_exact", "low")


In [ ]:
plot_regime("val/clean_final_exact", "low")


In [ ]:
plot_regime("val/clean_full_exact", "medium")


In [ ]:
plot_regime("val/clean_final_exact", "medium")


In [ ]:
plot_regime("val/clean_full_exact", "high")


In [ ]:
plot_regime("val/clean_final_exact", "high")


In [ ]:
# This view is the quick stitching sanity check for the resumed low-eta NAIL runs.
low_curve_df = load_curve_df_for_regime("low")
low_curve_df.loc[
    (low_curve_df["group"] == "NAIL-OPD MC") & (low_curve_df["eta"].isin([0.20, 0.30])),
    ["eta", "iter", "val/clean_full_exact", "val/clean_final_exact", "source", "source_priority"],
].sort_values(["eta", "iter", "source_priority"]).reset_index(drop=True)
